# GSL Baseline Comparison for Eviction Rate Prediction

Benchmarks 12 Graph Structure Learning methods against a fixed-graph GCN baseline
for census-tract-level eviction rate prediction.

**Temporal split:**
- **Train:** ACS 2022 features → predict eviction rate 2024
- **Test:**  ACS 2023 features → predict eviction rate 2025 (inductive — separate graph)

**Graph:** Cosine similarity kNN (K=1, threshold=0.3), symmetrised.

**Environment:** Tested on NVIDIA A100 40 GB PCIe (UTSA ARC).


## Configuration
Set all paths and hyperparameters here before running.

In [ ]:
import os

# ── Data paths — edit these to match your local setup ─────────────────────
ACS_DIR   = 'data/ACS_clean'      # ACS aligned CSV files (2022, 2023)
EVT_DIR   = 'data/Eviction'       # Eviction filing CSV files (2022–2025)
SAVE_DIR  = 'outputs/models'      # where trained model weights are saved
GRAPH_DIR = 'outputs/graphs'      # where learned adjacencies are saved

os.makedirs(SAVE_DIR,  exist_ok=True)
os.makedirs(GRAPH_DIR, exist_ok=True)

# ── Graph hyperparameters ─────────────────────────────────────────────────
SIM_K         = 1     # KNN neighbours for similarity graph
SIM_THRESHOLD = 0.3   # cosine distance threshold

# ── Training hyperparameters ──────────────────────────────────────────────
HIDDEN_DIM   = 64
EPOCHS       = 200
LR           = 1e-3
WEIGHT_DECAY = 5e-4
PATIENCE     = None    # early stopping patience (set None to disable)
VAL_FRAC     = 0.15   # fraction of train set used for validation
SEED         = 42

# ── Feature flags ─────────────────────────────────────────────────────────
USE_LAGGED = False    # set True to append lagged eviction rate as a feature


## Imports

In [ ]:
import argparse, math, time, warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import r2_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch.optim import Adam

from opengsl.module import GraphLearner
from opengsl.module.graphlearner import OneLayerNN
from opengsl.module.encoder import GCNEncoder, GCNDiagEncoder, AttentiveEncoder
from opengsl.module.metric import WeightedCosine, Cosine, InnerProduct, FGP
from opengsl.module.transform import KNN, EpsilonNN, NonLinear
from opengsl.module.fuse import Replace, Interpolate
from opengsl.module.functional import normalize
from opengsl.module.solver import GSLSolver
from opengsl.utils import set_seed

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## Data Loading

In [ ]:
def to_geoid_str(series: pd.Series) -> pd.Series:
    """Zero-pad GEOID to 11 characters."""
    return series.astype(str).str.strip().str.zfill(11)


def compute_eviction_rate(evt_df, acs_df, filings_col, pop_col, label):
    """Compute eviction rate = (filings / population) * 100."""
    pop = acs_df[[pop_col]].rename(columns={pop_col: 'population'})
    df  = evt_df[[filings_col]].copy()
    df['population'] = pop.reindex(df.index)['population']
    n_before = len(df)
    df = df[df['population'].notna() & (df['population'] > 0)]
    df[filings_col]     = df[filings_col].fillna(0)
    df['eviction_rate'] = (df[filings_col] / df['population']) * 100
    print(f'  {label}: {n_before:,} -> {len(df):,} tracts  '
          f'mean={df["eviction_rate"].mean():.4f}  max={df["eviction_rate"].max():.4f}')
    return df['eviction_rate']


print('Loading ACS data ...')
acs_2022 = pd.read_csv(f'{ACS_DIR}/acs_aligned_2022_5_tract.csv')
acs_2023 = pd.read_csv(f'{ACS_DIR}/acs_aligned_2023_5_tract.csv')
for df in [acs_2022, acs_2023]:
    df['FIPS'] = to_geoid_str(df['FIPS'])
    df.set_index('FIPS', inplace=True)
print(f'  ACS 2022 : {acs_2022.shape[0]:,} tracts, {acs_2022.shape[1]} features')
print(f'  ACS 2023 : {acs_2023.shape[0]:,} tracts, {acs_2023.shape[1]} features')

print('\nLoading eviction data ...')
FILINGS_COL    = 'total_filings'
POPULATION_COL = 'F001'
evt_2022 = pd.read_csv(f'{EVT_DIR}/eviction_filings_2022.csv')
evt_2023 = pd.read_csv(f'{EVT_DIR}/eviction_filings_2023.csv')
evt_2024 = pd.read_csv(f'{EVT_DIR}/eviction_filings_2024.csv')
evt_2025 = pd.read_csv(f'{EVT_DIR}/eviction_filings_2025.csv')
for df in [evt_2022, evt_2023, evt_2024, evt_2025]:
    df['GEOID'] = to_geoid_str(df['GEOID'])
    df.set_index('GEOID', inplace=True)

evt_rate_2022 = compute_eviction_rate(evt_2022, acs_2022, FILINGS_COL, POPULATION_COL, '2022 rate')
evt_rate_2023 = compute_eviction_rate(evt_2023, acs_2023, FILINGS_COL, POPULATION_COL, '2023 rate')
evt_rate_2024 = compute_eviction_rate(evt_2024, acs_2022, FILINGS_COL, POPULATION_COL, '2024 rate (train label)')
evt_rate_2025 = compute_eviction_rate(evt_2025, acs_2023, FILINGS_COL, POPULATION_COL, '2025 rate (test label)')


## Train / Test Split

In [ ]:
# Train: ACS 2022 -> predict eviction rate 2024
train_common = (acs_2022.index
                .intersection(evt_rate_2022.index)
                .intersection(evt_rate_2024.index))
train_X = acs_2022.loc[train_common].copy()
if USE_LAGGED:
    train_X['eviction_rate_prev'] = evt_rate_2022.reindex(train_common)
train_y = evt_rate_2024.reindex(train_common)
valid_train = train_y.notna()
if USE_LAGGED:
    valid_train &= train_X['eviction_rate_prev'].notna()
train_X = train_X[valid_train]
train_y = train_y[valid_train]

# Test: ACS 2023 -> predict eviction rate 2025
test_common = (acs_2023.index
               .intersection(evt_rate_2023.index)
               .intersection(evt_rate_2025.index))
test_X = acs_2023.loc[test_common].copy()
if USE_LAGGED:
    test_X['eviction_rate_prev'] = evt_rate_2023.reindex(test_common)
test_y = evt_rate_2025.reindex(test_common)
valid_test = test_y.notna()
if USE_LAGGED:
    valid_test &= test_X['eviction_rate_prev'].notna()
test_X = test_X[valid_test]
test_y = test_y[valid_test]

# Align feature columns across releases
if USE_LAGGED:
    acs_shared    = (train_X.columns.drop('eviction_rate_prev')
                     .intersection(test_X.columns.drop('eviction_rate_prev')).tolist())
    shared_feats  = acs_shared + ['eviction_rate_prev']
else:
    shared_feats  = train_X.columns.intersection(test_X.columns).tolist()
train_X = train_X[shared_feats]
test_X  = test_X[shared_feats]

print(f'Train : {len(train_X):,} tracts | Test : {len(test_X):,} tracts')
print(f'Features : {len(shared_feats)} (USE_LAGGED={USE_LAGGED})')


## Preprocessing and Graph Construction

In [ ]:
# Validation split (from train only)
rng      = np.random.default_rng(SEED)
idx      = rng.permutation(len(train_X))
n_val    = int(len(train_X) * VAL_FRAC)
val_idx  = idx[:n_val]
tr_idx   = idx[n_val:]

val_X   = train_X.iloc[val_idx];  val_y   = train_y.iloc[val_idx]
train_X = train_X.iloc[tr_idx];   train_y = train_y.iloc[tr_idx]
print(f'After val split — Train: {len(train_X):,}  Val: {len(val_X):,}  Test: {len(test_X):,}')

# Scale — fit on train only
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(train_X.values.astype(np.float32))
X_val_sc = scaler.transform(val_X.values.astype(np.float32))
X_te_sc  = scaler.transform(test_X.values.astype(np.float32))

N_train, D = X_tr_sc.shape
N_val      = X_val_sc.shape[0]
N_test     = X_te_sc.shape[0]
print(f'N_train={N_train:,}  N_val={N_val:,}  N_test={N_test:,}  D={D}')

# Labels
y_train = train_y.values.astype(np.float32)
y_val   = val_y.values.astype(np.float32)
y_test  = test_y.values.astype(np.float32)


def build_sim_adj(X_scaled, k=SIM_K, threshold=SIM_THRESHOLD):
    """
    Cosine kNN adjacency with distance threshold, then symmetrised.
    Returns dense float32 tensor [N, N].
    """
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric='cosine', algorithm='brute')
    nbrs.fit(X_scaled)
    dist, idx = nbrs.kneighbors(X_scaled)
    n = len(X_scaled)
    A = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for rank in range(1, k + 1):
            if dist[i, rank] < threshold:
                A[i, idx[i, rank]] = 1.0
    A = np.maximum(A, A.T)   # symmetrise
    return torch.tensor(A, dtype=torch.float32)


print('\nBuilding adjacency matrices ...')
adj_train = build_sim_adj(X_tr_sc).to(device)
adj_val   = build_sim_adj(X_val_sc).to(device)
adj_test  = build_sim_adj(X_te_sc).to(device)
print('Done')

# Move to device
X_train = torch.tensor(X_tr_sc,  dtype=torch.float32).to(device)
X_val   = torch.tensor(X_val_sc, dtype=torch.float32).to(device)
X_test  = torch.tensor(X_te_sc,  dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).to(device)


## Solver Infrastructure

In [ ]:
# ── Adapters ──────────────────────────────────────────────────────────────────

class OneLayerNNAdapter(nn.Module):
    """Wraps OneLayerNN (GLCN-style) to match GraphLearner signature."""
    def __init__(self, learner, fuse=None):
        super().__init__()
        self.learner = learner
        self.fuse_fn = fuse

    def forward(self, x, input_adj, raw_adj=None, return_mid=False):
        edge = (input_adj.coalesce().indices() if input_adj.is_sparse
                else input_adj.nonzero().t())
        adj_sp, _ = self.learner(x, edge, return_h=True)
        adj = adj_sp.to_dense()
        if self.fuse_fn is not None:
            adj = self.fuse_fn(adj, input_adj if raw_adj is None else raw_adj)
        return adj


class IdentityLearner(nn.Module):
    """Pass-through for vanilla GCN baseline (no graph modification)."""
    def __init__(self):
        super().__init__()
        self._dummy = nn.Parameter(torch.zeros(1))

    def forward(self, x, input_adj, raw_adj=None, return_mid=False):
        return input_adj


# ── Dataset wrapper ───────────────────────────────────────────────────────────

class EvictionDataset:
    """Wraps pre-built tensors into the interface expected by GSLSolver."""
    def __init__(self, X_tr, y_tr, X_va, y_va, X_te, y_te,
                 adj_tr, adj_va, adj_te, device):
        N_tr = X_tr.shape[0];  N_va = X_va.shape[0];  N_tv = N_tr + N_va
        self.feats  = torch.cat([X_tr, X_va], dim=0)
        self.labels = torch.cat([y_tr, y_va], dim=0)
        adj_tv = torch.zeros(N_tv, N_tv, device=device)
        adj_tv[:N_tr, :N_tr] = adj_tr
        adj_tv[N_tr:, N_tr:] = adj_va
        self.adj = adj_tv.to_sparse()
        tr_mask = torch.zeros(N_tv, dtype=torch.bool);  tr_mask[:N_tr] = True
        va_mask = torch.zeros(N_tv, dtype=torch.bool);  va_mask[N_tr:] = True
        self.train_masks = [tr_mask];  self.val_masks = [va_mask]
        self.test_masks  = [va_mask]   # placeholder; test is inductive
        self.n_nodes     = N_tv
        self.dim_feats   = X_tr.shape[1]
        self.num_targets = 1;  self.n_classes = 1
        self.name        = 'eviction'
        # Inductive test data
        self.feats_test  = X_te
        self.adj_test    = adj_te.to_sparse()
        self.labels_test = y_te


# ── EvictionGSLSolver ─────────────────────────────────────────────────────────

class EvictionGSLSolver(GSLSolver):
    """
    GSLSolver subclass for regression with inductive test evaluation.
    Supports early stopping on validation MSE and bin-wise RMSE reporting.
    """
    BINS = [(0,1,'0-1'),(1,5,'1-5'),(5,10,'5-10'),
            (10,20,'10-20'),(20,25,'20-25'),(25,float('inf'),'25+')]

    def __init__(self, conf, dataset, method_name='gsl', baseline=False):
        super().__init__(conf, dataset)
        self.method_name  = method_name
        self.baseline     = baseline
        self.feats_test   = dataset.feats_test
        self.adj_test     = (dataset.adj_test if conf.dataset['sparse']
                             else dataset.adj_test.to_dense())
        self.labels_test  = dataset.labels_test

    def _forward(self, feats, adj_sparse):
        if self.baseline:
            adj_norm = normalize(adj_sparse)
            return self.model(feats, adj_norm), adj_norm, None
        adj      = self.graphlearner(feats, normalize(adj_sparse))
        adj_norm = normalize(adj, add_loop=False)
        return self.model(feats, adj_norm), adj_norm, adj

    def learn(self, debug=False):
        for epoch in range(self.conf.training['n_epochs']):
            improve = ''
            t0 = time.time()
            self.model.train()
            if not self.baseline:
                self.graphlearner.train()
            self.optim.zero_grad()
            output, adj_norm, adj_raw = self._forward(self.feats, self.adj)
            loss_train = self.loss_fn(output[self.train_mask].squeeze(-1),
                                      self.labels[self.train_mask])
            loss_train.backward()
            params = list(self.model.parameters())
            if not self.baseline:
                params += list(self.graphlearner.parameters())
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            self.optim.step()
            loss_val, r2_val = self.evaluate(self.val_mask)
            flag, flag_stop = self.recoder.add(loss_val, r2_val)
            if flag:
                improve = '*'
                self.total_time      = time.time() - self.start_time
                self.best_val_loss   = loss_val
                self.result['valid'] = r2_val
                self.result['train'] = r2_score(
                    self.labels[self.train_mask].cpu().numpy(),
                    output[self.train_mask].detach().cpu().numpy())
                self.weights = deepcopy(self.model.state_dict())
                if not self.baseline:
                    self.gsl_weights       = deepcopy(self.graphlearner.state_dict())
                    self.adjs['final']     = adj_norm.detach().clone()
                    self.adjs['learned_train'] = adj_raw.detach().clone()
            elif flag_stop:
                if debug:
                    print(f'  Early stop @ epoch {epoch+1}')
                break
            if debug:
                print('Epoch {:04d} | {:.1f}s | Loss {:.4f} | '
                      'ValMSE {:.4f} | ValR2 {:.4f} | {}'.format(
                    epoch+1, time.time()-t0, loss_train.item(),
                    loss_val.item(), r2_val, improve))
        print('Optimization Finished!')
        print('Time(s): {:.4f}'.format(self.total_time))
        loss_test, r2_test = self.test()
        self.result['test'] = r2_test
        te_rmse, te_mae = self._regression_metrics()
        print(f'Test RMSE: {te_rmse:.4f} | MAE: {te_mae:.4f} | R2: {r2_test:.4f}')
        self._print_binwise()
        return self.result, self.adjs.get('final')

    def evaluate(self, mask):
        self.model.eval()
        if not self.baseline:
            self.graphlearner.eval()
        with torch.no_grad():
            output, _, _ = self._forward(self.feats, self.adj)
        preds  = output[mask].squeeze(-1)
        labels = self.labels[mask]
        loss   = self.loss_fn(preds, labels)
        r2     = r2_score(labels.cpu().numpy(), preds.cpu().numpy())
        return loss, r2

    def test(self):
        self.model.load_state_dict(self.weights)
        if not self.baseline:
            self.graphlearner.load_state_dict(self.gsl_weights)
        self.model.eval()
        if not self.baseline:
            self.graphlearner.eval()
        with torch.no_grad():
            output, _, adj_raw_test = self._forward(self.feats_test, self.adj_test)
        if adj_raw_test is not None:
            self.adjs['learned_test'] = adj_raw_test.detach().clone()
        preds  = output.squeeze(-1)
        labels = self.labels_test
        loss   = self.loss_fn(preds, labels)
        r2     = r2_score(labels.cpu().numpy(), preds.cpu().numpy())
        self._test_preds  = preds.cpu().numpy()
        self._test_labels = labels.cpu().numpy()
        return loss, r2

    def _regression_metrics(self):
        p, t = self._test_preds, self._test_labels
        return math.sqrt(((p-t)**2).mean()), np.abs(p-t).mean()

    def _print_binwise(self):
        p, t = self._test_preds, self._test_labels
        print(f'  {"Bin":>8}  {"N":>6}  {"RMSE":>8}')
        for lo, hi, label in self.BINS:
            mask = (t >= lo) & (t < hi)
            if mask.sum() == 0:
                continue
            rmse_b = math.sqrt(((p[mask]-t[mask])**2).mean())
            print(f'  {label:>8}  {mask.sum():>6,}  {rmse_b:>8.4f}')


## Solver Factory

In [ ]:
def build_eviction_solver(method_name, dataset, hidden_dim=HIDDEN_DIM,
                          lr=LR, wd=WEIGHT_DECAY,
                          epochs=EPOCHS, patience=PATIENCE):
    """
    Returns an EvictionGSLSolver configured for the named GSL method.

    Supported methods
    -----------------
    Baseline : gcn
    GSL      : grcn, idgl, gen, gaug, stable, nodeformer,
               segsl, cogsl, wsgnn, glcn, bmgcn, ggsl
    """
    dim_feats = dataset.dim_feats
    n_nodes   = dataset.n_nodes
    baseline  = (method_name == 'gcn')

    conf = argparse.Namespace(**{
        'use_deterministic': False,
        'model':    {'n_hidden': hidden_dim, 'n_layer': 2},
        'training': {'lr': lr, 'weight_decay': wd, 'n_epochs': epochs,
                     'patience': patience, 'criterion': 'loss'},
        'dataset':  {'feat_norm': False, 'sparse': True},
        'analysis': {'flag': False, 'save_graph': False},
    })

    METHOD_CONFIGS = {
        'gcn': lambda: IdentityLearner(),
        'grcn': lambda: GraphLearner(
            encoder=GCNDiagEncoder(2, dim_feats), metric=Cosine(),
            postprocess=[KNN(10)], fuse=Interpolate(1, 1)),
        'idgl': lambda: GraphLearner(
            metric=WeightedCosine(dim_feats, num_pers=4),
            postprocess=[EpsilonNN(0.65)], fuse=Interpolate(0.8, 0.2)),
        'gen': lambda: GraphLearner(
            metric=Cosine(), postprocess=[KNN(10)], fuse=Interpolate(0.5, 0.5)),
        'gaug': lambda: GraphLearner(
            encoder=GCNEncoder(dim_feats, nhid=hidden_dim, nclass=hidden_dim,
                               n_layers=2, dropout=0.0),
            metric=InnerProduct(), postprocess=[EpsilonNN(0.5)],
            fuse=Interpolate(0.8, 0.2)),
        'stable': lambda: GraphLearner(
            metric=WeightedCosine(dim_feats, num_pers=1),
            postprocess=[KNN(10)], fuse=Interpolate(0.5, 0.5)),
        'nodeformer': lambda: GraphLearner(
            metric=WeightedCosine(dim_feats, num_pers=4),
            postprocess=[KNN(10)], fuse=Replace()),
        'segsl': lambda: GraphLearner(
            metric=Cosine(), postprocess=[KNN(3)], fuse=Interpolate(0.5, 0.5)),
        'cogsl': lambda: GraphLearner(
            metric=Cosine(), postprocess=[KNN(3)], fuse=Interpolate(0.5, 0.5)),
        'wsgnn': lambda: GraphLearner(
            metric=WeightedCosine(dim_feats, num_pers=8),
            postprocess=[EpsilonNN(0.5)], fuse=Interpolate(0.5, 0.5)),
        'glcn': lambda: OneLayerNNAdapter(
            OneLayerNN(dim_feats, hidden_dim, p_dropout=0.5),
            fuse=Interpolate(0.8, 0.2)),
        'bmgcn': lambda: GraphLearner(
            metric=Cosine(), postprocess=[KNN(10)], fuse=Replace()),
        'ggsl': lambda: GraphLearner(
            encoder=GCNEncoder(dim_feats, nhid=hidden_dim, nclass=hidden_dim,
                               n_layers=2, dropout=0.5),
            metric=Cosine(), postprocess=[KNN(10)], fuse=Interpolate(0.5, 0.5)),
    }

    if method_name not in METHOD_CONFIGS:
        raise ValueError(
            f'Unknown method: {method_name}. '
            f'Choose from: {sorted(METHOD_CONFIGS)}')

    class _Solver(EvictionGSLSolver):
        def set_method(self):
            self.graphlearner = METHOD_CONFIGS[method_name]().to(self.device)
            self.model = GCNEncoder(
                dim_feats, nhid=hidden_dim, nclass=1,
                n_layers=2, dropout=0.5).to(self.device)
            param_groups = [{'params': self.model.parameters()}]
            if not self.baseline:
                param_groups.append({'params': self.graphlearner.parameters()})
            self.optim = Adam(param_groups, lr=lr, weight_decay=wd)

    return _Solver(conf, dataset, method_name=method_name, baseline=baseline)


## Run All Methods

In [ ]:
METHODS = [
    'gcn',        # Fixed-graph GCN baseline
    'grcn',       # Graph-Revised Convolutional Network
    'idgl',       # Iterative Deep Graph Learning
    'gen',        # Graph Estimation Network
    'gaug',       # Graph Augmentation
    'stable',     # STABLE (robust neighbour discovery)
    'nodeformer', # NodeFormer (kernel-based all-pair attention)
    'segsl',      # SEGSL (structural entropy)
    'cogsl',      # CoGSL (collaborative multi-view)
    'wsgnn',      # WSGNN (weakly supervised)
    'glcn',       # GLCN (neural edge reweighting)
    'bmgcn',      # BMGCN (block-model)
    'ggsl',       # GGSL (generalised GSL)
]

dataset = EvictionDataset(
    X_train, y_train_t, X_val, y_val_t, X_test, y_test_t,
    adj_train, adj_val, adj_test, device=device)

results = {}
for method in METHODS:
    set_seed(SEED)
    print(f"\n{'='*60}")
    print(f'  Method: {method.upper()}')
    print(f"{'='*60}")
    solver = build_eviction_solver(method, dataset)
    solver.set(split=0)
    result, final_adj = solver.learn(debug=True)
    results[method] = result

    # Save model weights
    save_dict = {'method': method, 'result': result,
                 'model_weights': solver.weights}
    if not solver.baseline:
        save_dict['gsl_weights'] = solver.gsl_weights
    torch.save(save_dict, os.path.join(SAVE_DIR, f'{method}.pt'))

    # Save learned graphs
    graph_dict = {'method': method}
    if solver.adjs.get('learned_train') is not None:
        graph_dict['train_adj'] = solver.adjs['learned_train'].cpu().to_sparse()
    if solver.adjs.get('learned_test') is not None:
        graph_dict['test_adj'] = solver.adjs['learned_test'].cpu().to_sparse()
    if len(graph_dict) > 1:
        torch.save(graph_dict, os.path.join(GRAPH_DIR, f'learned_graphs_{method}.pt'))


## Results Summary

In [ ]:
import pandas as pd

rows = []
for method, result in results.items():
    rows.append({
        'Method':     method,
        'Test RMSE':  result.get('test_rmse', float('nan')),
        'Test MAE':   result.get('test_mae',  float('nan')),
        'Test R2':    result.get('test',      float('nan')),
    })

summary = pd.DataFrame(rows).sort_values('Test RMSE').reset_index(drop=True)
print(summary.to_string(index=False))
